# Benchmark on a real NVIDIA GPU

Run this in Google Colab with a GPU runtime: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

It clones the repo, checks every kernel against `torch.matmul`, times it, and saves a CSV you can download.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/kajalhoskul/triton-gemm-from-scratch.git
%cd triton-gemm-from-scratch
!git log --oneline -1

In [ ]:
# Colab already ships with PyTorch and Triton; this only adds anything missing.
!pip install -q -r requirements.txt
import torch, triton
print('torch', torch.__version__, '| triton', triton.__version__, '|', torch.cuda.get_device_name())

In [ ]:
!python -m pytest -q

## Benchmark

**FP32** compares steps 1 to 3 on equal terms: the T4 has no TF32, so cuBLAS and Triton both use ordinary FP32 arithmetic. The naive kernel is very slow, so it runs on smaller sizes only.

**FP16** is step 4: `tl.dot` on FP16 tiles uses the T4's Tensor Cores.

In [ ]:
# FP32: every step side by side
!python -m gemm.bench --kernels torch tiled autotuned --dtype float32 --sizes 512 1024 2048 4096 --out results/t4_fp32.csv
!python -m gemm.bench --kernels naive --dtype float32 --sizes 256 512 1024 --warmup 2 --reps 5 --out results/t4_fp32_naive.csv

In [ ]:
# FP16 with Tensor Cores (step 4)
!python -m gemm.bench --kernels torch tiled autotuned grouped fast --dtype float16 --sizes 512 1024 2048 4096 --out results/t4_fp16.csv

In [ ]:
# Roofline: the best the T4 could do for each shape, and how close each kernel gets
!python -m gemm.roofline results/t4_fp32.csv results/t4_fp16.csv

In [ ]:
# Which tile size did each autotuner pick?
from gemm.bench import make_inputs
from gemm import tiled, fast
for s in (512, 1024, 2048, 4096):
    a, b = make_inputs(s, s, s, dtype=torch.float32, device='cuda')
    tiled.autotuned_matmul(a, b)
    a16, b16 = a.half(), b.half()
    fast.fast_matmul(a16, b16)
    print(s, 'fp32 tiled:', tiled.best_config(s, s, s), '| fp16 fast:', fast.best_config(s, s, s))

In [ ]:
from google.colab import files
for name in ['t4_fp32.csv', 't4_fp32_naive.csv', 't4_fp16.csv']:
    files.download(f'results/{name}')